# Human Ratings

Calculate the agreement:
- Between human raters (Intra);
- Between human raters and LLM-judges (Inter);


In [1]:
import pandas as pd

r1 = pd.read_csv("../data/ratings_F_28.csv")
r2 = pd.read_csv("../data/ratings_F_63.csv")
r3 = pd.read_csv("../data/ratings_M_30.csv")
r4 = pd.read_csv("../data/ratings_M_32.csv")\

r1["origem"] = "F_28"
r2["origem"] = "F_63"
r3["origem"] = "M_30"
r4["origem"] = "M_32"

df = pd.concat([r1, r2, r3, r4], ignore_index=True)

r1

,n_record,academic_simple,academic_detailed,pitch_simple,pitch_detailed,origem
0,0,3,2,4,1,F_28
1,1,3,2,5,1,F_28
2,2,4,1,5,2,F_28
3,3,3,1,5,2,F_28
4,4,5,2,5,1,F_28
5,5,4,1,4,3,F_28
6,6,4,1,5,3,F_28
7,7,3,1,4,2,F_28
8,8,4,2,5,1,F_28
9,9,4,1,5,2,F_28


In [2]:
import json

## function to process results from scalar and pairwise evaluations into two separate dataframes
def process_results(file_path):
    all_rows = []
    pairwise_rows = []

    comparison_labels = [
        ("academic_simple", "academic_detailed"), # Index 0
        ("pitch_simple", "pitch_detailed"),       # Index 1
        ("academic_simple", "pitch_simple"),       # Index 2
        ("academic_detailed", "pitch_detailed")    # Index 3
    ]
    
    with open(file_path, "r") as f:
        for line in f:
            record = json.loads(line)

            # record contains: index, event, title, category, scalar, pairwise
            # each 'scalar' entry has 4 conditions: academic_simple, academic_detailed, etc.
            for cond_name, metrics in record["scalar"].items():
        
                if metrics:
                    # split condition name (e.g., 'academic_simple') into persona and complexity
                    persona, complexity = cond_name.split("_")
                    
                    all_rows.append({
                        'presentation_id': record["index"], # using index as ID
                        'type': record["category"],        # presentation type
                        'persona': persona,
                        'complexity': complexity,
                        'grounding': metrics.get('grounding'),
                        'actionability': metrics.get('actionability'),
                        'persona_consistency': metrics.get('persona_consistency'),
                        'clarity': metrics.get('clarity'),
                        'explanation': metrics.get('explanation')
                    })

            for i, res in enumerate(record["pairwise"]):
                if res and i < len(comparison_labels):
                    cond_a, cond_b = comparison_labels[i]
                    pairwise_rows.append({
                        'presentation_id': record["index"],
                        'presentation_type': record["category"],
                        'comparison': f"{cond_a}_vs_{cond_b}",
                        'model_a': cond_a,
                        'model_b': cond_b,
                        'better_grounding': res.get('better_grounding'),
                        'better_actionability': res.get('better_actionability'),
                        'better_clarity': res.get('better_clarity'),
                        'better_overall': res.get('better_overall'),
                        'explanation': res.get('explanation')
                    })
    
    return pd.DataFrame(all_rows), pd.DataFrame(pairwise_rows)



# load and transform
df_scale, df_pair = process_results("../data/llm_judge_results_gpt41.jsonl")
df_scale4o, df_pair4o = process_results("../data/llm_judge_results_gpt4o.jsonl")
df_scale['avg_score'] = df_scale[['grounding', 'actionability', 'persona_consistency', 'clarity']].mean(axis=1)
df_scale4o['avg_score'] = df_scale4o[['grounding', 'actionability', 'persona_consistency', 'clarity']].mean(axis=1)
df_scale['condition'] = df_scale['persona'] + "_" + df_scale['complexity']
df_scale4o['condition'] = df_scale4o['persona'] + "_" + df_scale4o['complexity']

df_scale

,presentation_id,type,persona,complexity,grounding,actionability,persona_consistency,clarity,explanation,avg_score,condition
0,0,academic,academic,simple,4.7,4.5,4.9,4.8,The feedback is deeply grounded in the emotion...,4.725,academic_simple
1,0,academic,academic,detailed,4.7,4.5,5.0,4.8,The feedback is strongly grounded in the emoti...,4.750,academic_detailed
2,0,academic,pitch,simple,4.8,4.9,5.0,4.8,The feedback is deeply grounded in the emotion...,4.875,pitch_simple
3,0,academic,pitch,detailed,4.8,4.7,5.0,4.9,The feedback is tightly aligned with the emoti...,4.850,pitch_detailed
4,1,academic,academic,simple,4.8,4.7,5.0,4.9,The feedback is deeply grounded in the emotion...,4.850,academic_simple
...,...,...,...,...,...,...,...,...,...,...,...
91,22,pitch,pitch,detailed,4.7,4.8,5.0,5.0,The feedback is strongly grounded in the emoti...,4.875,pitch_detailed
92,23,pitch,academic,simple,4.8,4.6,5.0,4.9,The feedback is deeply grounded in the emotion...,4.825,academic_simple
93,23,pitch,academic,detailed,4.8,4.7,4.9,5.0,The feedback is deeply grounded in the emotion...,4.850,academic_detailed
94,23,pitch,pitch,simple,4.8,4.9,5.0,5.0,The feedback is deeply grounded in the emotion...,4.925,pitch_simple


In [3]:
df_scale4o

,presentation_id,type,persona,complexity,grounding,actionability,persona_consistency,clarity,explanation,avg_score,condition
0,0,academic,academic,simple,4.0,4.5,4.5,4.5,The feedback aligns well with the emotion time...,4.375,academic_simple
1,0,academic,academic,detailed,4.0,4.5,5.0,4.5,The feedback aligns well with the emotion time...,4.500,academic_detailed
2,0,academic,pitch,simple,4.0,4.5,5.0,4.5,The feedback aligns well with the emotion time...,4.500,pitch_simple
3,0,academic,pitch,detailed,4.5,4.0,5.0,4.5,The feedback aligns well with the emotion time...,4.500,pitch_detailed
4,1,academic,academic,simple,4.5,4.0,5.0,4.5,The feedback is well-grounded in the emotion t...,4.500,academic_simple
...,...,...,...,...,...,...,...,...,...,...,...
91,22,pitch,pitch,detailed,4.0,4.5,5.0,4.5,The feedback aligns well with the emotion time...,4.500,pitch_detailed
92,23,pitch,academic,simple,4.0,4.5,4.5,4.5,The feedback aligns well with the emotion time...,4.375,academic_simple
93,23,pitch,academic,detailed,4.5,4.0,4.5,4.5,The feedback aligns well with the emotion time...,4.375,academic_detailed
94,23,pitch,pitch,simple,4.5,4.8,5.0,4.7,The feedback is well-grounded in the emotion t...,4.750,pitch_simple


In [4]:
!pip install pingouin

In [5]:
import pandas as pd
import pingouin as pg
from scipy.stats import pearsonr, spearmanr

############################################################
# 1. Read human ratings
############################################################

r1 = pd.read_csv("../data/ratings_F_28.csv")
r2 = pd.read_csv("../data/ratings_F_63.csv")
r3 = pd.read_csv("../data/ratings_M_30.csv")
r4 = pd.read_csv("../data/ratings_M_32.csv")

r1["annotator"] = "F_28"
r2["annotator"] = "F_63"
r3["annotator"] = "M_30"
r4["annotator"] = "M_32"

df_human = pd.concat([r1, r2, r3, r4], ignore_index=True)

############################################################
# 2. Long format
############################################################

human_long = df_human.melt(
    id_vars=["n_record", "annotator"],
    value_vars=[
        "academic_simple",
        "academic_detailed",
        "pitch_simple",
        "pitch_detailed"
    ],
    var_name="condition",
    value_name="human_score"
)

human_long.rename(
    columns={"n_record": "presentation_id"},
    inplace=True
)

human_long["item"] = (
    human_long["presentation_id"].astype(str)
    + "_"
    + human_long["condition"]
)

############################################################
# 3. ICC between the four annotators
############################################################

icc = pg.intraclass_corr(
    data=human_long,
    targets="item",
    raters="annotator",
    ratings="human_score"
)

print("\n========== ICC ==========\n")
print(icc)

# ICC(2,1) is usually reported for independent raters
icc2 = icc[icc["Type"] == "ICC2"]

print("\nReported ICC:")
print(icc2[["Type","ICC","CI95","F","pval"]])

############################################################
# 4. Average human score
############################################################

human_mean = (
    human_long
    .groupby(
        ["presentation_id","condition"],
        as_index=False
    )["human_score"]
    .mean()
)

############################################################
# 5. LLM scores
############################################################

df_scale, df_pair = process_results("../data/llm_judge_results_gpt41.jsonl")
df_scale4o, df_pair4o = process_results("../data/llm_judge_results_gpt4o.jsonl")

for df in [df_scale, df_scale4o]:

    df["avg_score"] = df[
        [
            "grounding",
            "actionability",
            "persona_consistency",
            "clarity"
        ]
    ].mean(axis=1)

    df["condition"] = (
        df["persona"] + "_" + df["complexity"]
    )

############################################################
# 6. Merge humans with GPT-4.1
############################################################

comparison41 = pd.merge(
    df_scale[
        ["presentation_id","condition","avg_score"]
    ],
    human_mean,
    on=["presentation_id","condition"]
)

############################################################
# 7. Merge humans with GPT-4o
############################################################

comparison4o = pd.merge(
    df_scale4o[
        ["presentation_id","condition","avg_score"]
    ],
    human_mean,
    on=["presentation_id","condition"]
)

############################################################
# 8. Consensus score
############################################################

consensus = pd.merge(
    df_scale[
        ["presentation_id","condition","avg_score"]
    ],
    df_scale4o[
        ["presentation_id","condition","avg_score"]
    ],
    on=["presentation_id","condition"],
    suffixes=("_41","_4o")
)

consensus["avg_score"] = (
    consensus["avg_score_41"] +
    consensus["avg_score_4o"]
) / 2

comparison_consensus = pd.merge(
    consensus[
        ["presentation_id","condition","avg_score"]
    ],
    human_mean,
    on=["presentation_id","condition"]
)

############################################################
# 9. Pearson correlations
############################################################

def report(name, df):

    r, p = pearsonr(
        df["avg_score"],
        df["human_score"]
    )

    rho, ps = spearmanr(
        df["avg_score"],
        df["human_score"]
    )

    print("\n----------------------------")
    print(name)
    print("----------------------------")
    print(f"Pearson  r = {r:.3f} (p={p:.4f})")
    print(f"Spearman ρ = {rho:.3f} (p={ps:.4f})")

report("GPT-4.1 vs Humans", comparison41)
report("GPT-4o vs Humans", comparison4o)
report("Consensus vs Humans", comparison_consensus)


========== ICC ==========

       Type       ICC         F  df1  df2          pval          CI95
0  ICC(1,1)  0.386275  3.517581   95  288  1.668863e-16   [0.28, 0.5]
1  ICC(A,1)  0.422762  5.981972   95  285  4.227882e-32  [0.22, 0.59]
2  ICC(C,1)  0.554664  5.981972   95  285  4.227882e-32  [0.46, 0.65]
3  ICC(1,k)  0.715714  3.517581   95  288  1.668863e-16   [0.61, 0.8]
4  ICC(A,k)  0.745518  5.981972   95  285  4.227882e-32  [0.53, 0.85]
5  ICC(C,k)  0.832831  5.981972   95  285  4.227882e-32  [0.77, 0.88]

Reported ICC:
Empty DataFrame
Columns: [Type, ICC, CI95, F, pval]
Index: []

----------------------------
GPT-4.1 vs Humans
----------------------------
Pearson  r = -0.013 (p=0.9038)
Spearman ρ = 0.142 (p=0.1672)

----------------------------
GPT-4o vs Humans
----------------------------
Pearson  r = 0.026 (p=0.8047)
Spearman ρ = 0.077 (p=0.4553)

----------------------------
Consensus vs Humans
----------------------------
Pearson  r = 0.015 (p=0.8883)
Spearman ρ = 0.119 (p=

In [6]:
r1

,n_record,academic_simple,academic_detailed,pitch_simple,pitch_detailed,annotator
0,0,3,2,4,1,F_28
1,1,3,2,5,1,F_28
2,2,4,1,5,2,F_28
3,3,3,1,5,2,F_28
4,4,5,2,5,1,F_28
5,5,4,1,4,3,F_28
6,6,4,1,5,3,F_28
7,7,3,1,4,2,F_28
8,8,4,2,5,1,F_28
9,9,4,1,5,2,F_28


In [7]:
comparison_consensus

,presentation_id,condition,avg_score,human_score
0,0,academic_simple,4.5500,4.25
1,0,academic_detailed,4.6250,3.00
2,0,pitch_simple,4.6875,4.50
3,0,pitch_detailed,4.6750,3.00
4,1,academic_simple,4.6750,3.75
...,...,...,...,...
91,22,pitch_detailed,4.6875,3.00
92,23,academic_simple,4.6000,4.75
93,23,academic_detailed,4.6125,3.00
94,23,pitch_simple,4.8375,4.50


In [8]:
import numpy as np

llm = comparison_consensus["avg_score"]
human = comparison_consensus["human_score"]

comparison_consensus["llm_calibrated"] = (
    (llm - llm.mean())
    /
    llm.std()
    *
    human.std()
    +
    human.mean()
)

rho, p = spearmanr(
    comparison_consensus["llm_calibrated"],
    human
)

print(rho,p)

0.11856575349348943 0.24991809390482883


In [9]:
icc_llm = pg.intraclass_corr(
    data=comparison_consensus.melt(
        id_vars=["presentation_id","condition"],
        value_vars=["avg_score","human_score"],
        var_name="rater",
        value_name="score"
    ),
    targets="presentation_id",
    raters="rater",
    ratings="score"
)

print(icc_llm)

       Type        ICC         F  df1  df2      pval             CI95
0  ICC(1,1)  -0.904180  0.050321   23   24  1.000000   [-0.96, -0.79]
1  ICC(A,1)  -0.004251  0.855979   23   23  0.643819    [-0.02, 0.03]
2  ICC(C,1)  -0.077599  0.855979   23   23  0.643819    [-0.46, 0.33]
3  ICC(1,k) -18.872432  0.050321   23   24  1.000000  [-44.35, -7.64]
4  ICC(A,k)  -0.008539  0.855979   23   23  0.643819    [-0.04, 0.07]
5  ICC(C,k)  -0.168253  0.855979   23   23  0.643819     [-1.7, 0.49]


In [10]:
df_human

,n_record,academic_simple,academic_detailed,pitch_simple,pitch_detailed,annotator
0,0,3,2,4,1,F_28
1,1,3,2,5,1,F_28
2,2,4,1,5,2,F_28
3,3,3,1,5,2,F_28
4,4,5,2,5,1,F_28
...,...,...,...,...,...,...
91,19,4,4,5,3,M_32
92,20,4,3,5,3,M_32
93,21,4,3,5,3,M_32
94,22,4,3,5,3,M_32


In [11]:
human_long

,presentation_id,annotator,condition,human_score,item
0,0,F_28,academic_simple,3,0_academic_simple
1,1,F_28,academic_simple,3,1_academic_simple
2,2,F_28,academic_simple,4,2_academic_simple
3,3,F_28,academic_simple,3,3_academic_simple
4,4,F_28,academic_simple,5,4_academic_simple
...,...,...,...,...,...
379,19,M_32,pitch_detailed,3,19_pitch_detailed
380,20,M_32,pitch_detailed,3,20_pitch_detailed
381,21,M_32,pitch_detailed,3,21_pitch_detailed
382,22,M_32,pitch_detailed,3,22_pitch_detailed


In [12]:
df_scale

,presentation_id,type,persona,complexity,grounding,actionability,persona_consistency,clarity,explanation,avg_score,condition
0,0,academic,academic,simple,4.7,4.5,4.9,4.8,The feedback is deeply grounded in the emotion...,4.725,academic_simple
1,0,academic,academic,detailed,4.7,4.5,5.0,4.8,The feedback is strongly grounded in the emoti...,4.750,academic_detailed
2,0,academic,pitch,simple,4.8,4.9,5.0,4.8,The feedback is deeply grounded in the emotion...,4.875,pitch_simple
3,0,academic,pitch,detailed,4.8,4.7,5.0,4.9,The feedback is tightly aligned with the emoti...,4.850,pitch_detailed
4,1,academic,academic,simple,4.8,4.7,5.0,4.9,The feedback is deeply grounded in the emotion...,4.850,academic_simple
...,...,...,...,...,...,...,...,...,...,...,...
91,22,pitch,pitch,detailed,4.7,4.8,5.0,5.0,The feedback is strongly grounded in the emoti...,4.875,pitch_detailed
92,23,pitch,academic,simple,4.8,4.6,5.0,4.9,The feedback is deeply grounded in the emotion...,4.825,academic_simple
93,23,pitch,academic,detailed,4.8,4.7,4.9,5.0,The feedback is deeply grounded in the emotion...,4.850,academic_detailed
94,23,pitch,pitch,simple,4.8,4.9,5.0,5.0,The feedback is deeply grounded in the emotion...,4.925,pitch_simple


In [15]:
import pandas as pd
import pingouin as pg
from scipy.stats import pearsonr, spearmanr


############################################################
# 1. Load human ratings
############################################################

files = {
    "F_28": "../data/ratings_F_28.csv",
    "F_63": "../data/ratings_F_63.csv",
    "M_30": "../data/ratings_M_30.csv",
    "M_32": "../data/ratings_M_32.csv"
}

human_list = []

for annotator, path in files.items():

    tmp = pd.read_csv(path)
    tmp["annotator"] = annotator
    
    human_list.append(tmp)


df_human = pd.concat(
    human_list,
    ignore_index=True
)


############################################################
# 2. Convert humans to long format
############################################################

conditions = [
    "academic_simple",
    "academic_detailed",
    "pitch_simple",
    "pitch_detailed"
]


human_long = df_human.melt(
    id_vars=[
        "n_record",
        "annotator"
    ],
    value_vars=conditions,
    var_name="condition",
    value_name="human_score"
)


human_long.rename(
    columns={
        "n_record": "presentation_id"
    },
    inplace=True
)


human_long["item"] = (
    human_long["presentation_id"].astype(str)
    + "_"
    + human_long["condition"]
)


print(human_long.head())


############################################################
# 3. Human inter-rater reliability
############################################################

icc = pg.intraclass_corr(
    data=human_long,
    targets="item",
    raters="annotator",
    ratings="human_score"
)


print("\n========== HUMAN ICC ==========")
print(icc)


# Average-measures absolute agreement ICC
print(
    "\nICC(A,k):"
)

print(
    icc[
        icc["Type"]=="ICC(A,k)"
    ][
        [
            "Type",
            "ICC",
            "CI95",
            "F",
            "pval"
        ]
    ]
)



############################################################
# 4. Mean human score
############################################################

human_mean = (
    human_long
    .groupby(
        [
            "presentation_id",
            "condition"
        ],
        as_index=False
    )
    ["human_score"]
    .mean()
)


print(human_mean.head())



############################################################
# 5. Load LLM scores
############################################################

df_scale, _ = process_results(
    "../data/llm_judge_results_gpt41.jsonl"
)

df_scale4o, _ = process_results(
    "../data/llm_judge_results_gpt4o.jsonl"
)


for df in [
    df_scale,
    df_scale4o
]:

    df["avg_score"] = df[
        [
            "grounding",
            "actionability",
            "persona_consistency",
            "clarity"
        ]
    ].mean(axis=1)


    df["condition"] = (
        df["persona"]
        +
        "_"
        +
        df["complexity"]
    )



############################################################
# 6. Merge LLM + Humans
############################################################

comparison41 = pd.merge(
    df_scale[
        [
            "presentation_id",
            "condition",
            "avg_score"
        ]
    ],
    human_mean,
    on=[
        "presentation_id",
        "condition"
    ],
    how="inner"
)


comparison4o = pd.merge(
    df_scale4o[
        [
            "presentation_id",
            "condition",
            "avg_score"
        ]
    ],
    human_mean,
    on=[
        "presentation_id",
        "condition"
    ],
    how="inner"
)



############################################################
# 7. LLM consensus
############################################################

consensus = pd.merge(
    df_scale[
        [
            "presentation_id",
            "condition",
            "avg_score"
        ]
    ],
    df_scale4o[
        [
            "presentation_id",
            "condition",
            "avg_score"
        ]
    ],
    on=[
        "presentation_id",
        "condition"
    ],
    suffixes=(
        "_gpt41",
        "_gpt4o"
    )
)


consensus["avg_score"] = (
    consensus["avg_score_gpt41"]
    +
    consensus["avg_score_gpt4o"]
) / 2



comparison_consensus = pd.merge(
    consensus[
        [
            "presentation_id",
            "condition",
            "avg_score"
        ]
    ],
    human_mean,
    on=[
        "presentation_id",
        "condition"
    ],
    how="inner"
)


print(
    "Number of comparisons:",
    len(comparison_consensus)
)



############################################################
# 8. Correlations
############################################################

def report(name, df):

    pearson_r, pearson_p = pearsonr(
        df["avg_score"],
        df["human_score"]
    )


    spearman_rho, spearman_p = spearmanr(
        df["avg_score"],
        df["human_score"]
    )


    print("\n================")
    print(name)
    print("================")
    print(
        f"Pearson r={pearson_r:.3f}, p={pearson_p:.4f}"
    )
    print(
        f"Spearman rho={spearman_rho:.3f}, p={spearman_p:.4f}"
    )


report(
    "GPT-4.1",
    comparison41
)

report(
    "GPT-4o",
    comparison4o
)

report(
    "Consensus",
    comparison_consensus
)

   presentation_id annotator        condition  human_score               item
0                0      F_28  academic_simple            3  0_academic_simple
1                1      F_28  academic_simple            3  1_academic_simple
2                2      F_28  academic_simple            4  2_academic_simple
3                3      F_28  academic_simple            3  3_academic_simple
4                4      F_28  academic_simple            5  4_academic_simple

========== HUMAN ICC ==========
       Type       ICC         F  df1  df2          pval          CI95
0  ICC(1,1)  0.386275  3.517581   95  288  1.668863e-16   [0.28, 0.5]
1  ICC(A,1)  0.422762  5.981972   95  285  4.227882e-32  [0.22, 0.59]
2  ICC(C,1)  0.554664  5.981972   95  285  4.227882e-32  [0.46, 0.65]
3  ICC(1,k)  0.715714  3.517581   95  288  1.668863e-16   [0.61, 0.8]
4  ICC(A,k)  0.745518  5.981972   95  285  4.227882e-32  [0.53, 0.85]
5  ICC(C,k)  0.832831  5.981972   95  285  4.227882e-32  [0.77, 0.88]

ICC(A,k)

In [33]:
expected = human_mean.merge(
    df_scale[["presentation_id","condition"]],
    on=["presentation_id","condition"],
    how="left",
    indicator=True
)

print(
    expected[
        expected["_merge"]=="left_only"
    ]
)

    presentation_id          condition  human_score     _merge
68               17  academic_detailed         2.50  left_only
69               17    academic_simple         4.25  left_only
70               17     pitch_detailed         2.75  left_only
71               17       pitch_simple         4.00  left_only
72               18  academic_detailed         3.00  left_only
73               18    academic_simple         4.75  left_only
74               18     pitch_detailed         2.75  left_only
75               18       pitch_simple         4.50  left_only
76               19  academic_detailed         3.50  left_only
77               19    academic_simple         4.25  left_only
78               19     pitch_detailed         2.50  left_only
79               19       pitch_simple         4.50  left_only
84               21  academic_detailed         3.00  left_only
85               21    academic_simple         4.50  left_only
86               21     pitch_detailed         3.00  le

In [34]:
expected2 = df_scale.merge(
    human_mean,
    on=["presentation_id","condition"],
    how="left",
    indicator=True
)

print(
    expected2[
        expected2["_merge"]=="left_only"
    ]
)

    presentation_id   type   persona complexity  grounding  actionability  \
80               24  pitch  academic     simple        4.8            4.7   
81               24  pitch  academic   detailed        4.8            4.7   
82               24  pitch     pitch     simple        4.9            4.8   
83               24  pitch     pitch   detailed        4.8            4.7   
84               25  pitch  academic     simple        4.7            4.5   
85               25  pitch  academic   detailed        4.7            4.6   
86               25  pitch     pitch     simple        4.8            4.7   
87               25  pitch     pitch   detailed        4.8            4.7   
88               26  pitch  academic     simple        4.7            4.3   
89               26  pitch  academic   detailed        4.8            4.7   
90               26  pitch     pitch     simple        4.9            4.8   
91               26  pitch     pitch   detailed        4.7            4.8   

In [35]:
print(expected[expected["_merge"]=="left_only"])

print(expected2[expected2["_merge"]=="left_only"])

    presentation_id          condition  human_score     _merge
68               17  academic_detailed         2.50  left_only
69               17    academic_simple         4.25  left_only
70               17     pitch_detailed         2.75  left_only
71               17       pitch_simple         4.00  left_only
72               18  academic_detailed         3.00  left_only
73               18    academic_simple         4.75  left_only
74               18     pitch_detailed         2.75  left_only
75               18       pitch_simple         4.50  left_only
76               19  academic_detailed         3.50  left_only
77               19    academic_simple         4.25  left_only
78               19     pitch_detailed         2.50  left_only
79               19       pitch_simple         4.50  left_only
84               21  academic_detailed         3.00  left_only
85               21    academic_simple         4.50  left_only
86               21     pitch_detailed         3.00  le